# Session 4 - Indexes and Query Optimization Basics

Welcome. In this session we will learn:

- what an **index** is for (and when it helps)
- how to read **`EXPLAIN`**
- how **`EXPLAIN ANALYZE`** shows real timings
- **sargable** filters (filters that can use an index cleanly)

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - What is an index?

## Phone-book story

Finding "Sharma" in a phone book:

```
No index (table scan)              With an index (lookup)
----------------------             ----------------------
Open page 1, scan names            Open index: S -> page 412
Open page 2, scan names            Go straight to page 412
... keep going ...                 Done
```

MySQL stores table rows on disk. An **index** is a sorted shortcut (usually a B-tree) that points to matching rows.

## Diagram: table vs index

```
training.orders (table heap / data)     idx on customer_id (sorted shortcut)
+-----------+--------+--------+         +-------------+-----------+
| order_id  | cust   | date   |         | customer_id | -> row    |
+-----------+--------+--------+         +-------------+-----------+
| 10248     | 1001   | 2023.. |         | 1001        | -> 10248  |
| 10249     | 2044   | 2023.. |         | 1001        | -> 11002  |
| 10250     | 1001   | 2023.. |         | 1002        | -> 10255  |
+-----------+--------+--------+         +-------------+-----------+

Query: WHERE customer_id = 1001
  -> use index, fetch only matching order rows
```

## Trade-off (keep this small)

| Upside | Cost |
|--------|------|
| Faster filters / joins on that column | Insert/update/delete must update the index |
| Less data read when selective | Uses disk space |

**When indexes help**

| Filter style | Typical plan | Why |
|--------------|--------------|-----|
| Selective (`customer_id = 1001`) | Index lookup | few matching rows |
| Broad (`freight > 50`) | Often full scan | too many rows still match |

Selective filters love indexes. Broad filters may still scan the table - and that can be the right choice.


---
# Chapter 2 - Indexes already on these tables

Useful indexes were created when the lab database was built (keys like `customer_id`, `order_date`, and foreign keys).

We **feel** them through `EXPLAIN` / `EXPLAIN ANALYZE` next - no need to invent new permanent indexes on `training.*`.

In **Chapter 6** you will also `SHOW` / `CREATE` / `DROP` an index on a temporary table.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Indexes</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - Reading EXPLAIN

`EXPLAIN` shows the **plan** MySQL chooses - it does **not** run the full query just to time it.

Think of it as asking the GPS for the route before you drive.

## Columns worth reading first

| Column | Simple meaning |
|--------|----------------|
| `type` | How rows are accessed (`const`, `ref`, `range`, `index`, `ALL`, ...) |
| `key` | Which index was chosen (`NULL` = no index used) |
| `rows` | Rough estimate of rows MySQL expects to examine |
| `Extra` | Notes like `Using where`, `Using index` |

## Access types (short map)

```
const / ref / range  ->  usually "used an index / tight access"
index                ->  walked an index (not always bad)
ALL                  ->  read the table (full scan)
```

None of these is always wrong. A full scan can be fine for small tables or broad filters.


### Lookup by primary key (expect a tight plan)


**Story:** Look up one customer by id - like opening one locker with the exact key.


In [1]:
EXPLAIN
SELECT *
FROM training.customers
WHERE customer_id = 1001;


+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
| id | select_type | table     | partitions | type  | possible_keys | key     | key_len | ref   | rows | filtered | Extra |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | customers | NULL       | const | PRIMARY       | PRIMARY | 4       | const |    1 |   100.00 | NULL  |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+


### Filter on an indexed date column


**Use case:** "Show me orders on one date." An index on `order_date` can jump to that date.


In [2]:
EXPLAIN
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
| id | select_type | table  | partitions | type | possible_keys         | key                   | key_len | ref   | rows | filtered | Extra |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | orders | NULL       | ref  | idx_orders_order_date | idx_orders_order_date | 3       | const |   57 |   100.00 | NULL  |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+


### Broad filter (often a table scan)


**Story:** "Show freight over 50" may match a huge share of orders.
Scanning the table can be cheaper than bouncing through an index for almost every row.


In [3]:
EXPLAIN
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
| id | select_type | table  | partitions | type | possible_keys | key  | key_len | ref  | rows  | filtered | Extra       |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
|  1 | SIMPLE      | orders | NULL       | ALL  | NULL          | NULL | NULL    | NULL | 38996 |    33.33 | Using where |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+


### What to look for in the plan

1. Does `key` show an index name, or `NULL`?
2. Is `type` a tight access (`const` / `ref` / `range`) or `ALL`?
3. Is `rows` small or huge?

Compare the three plans above: primary-key lookup, indexed date, broad freight filter.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - EXPLAIN</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 4 - EXPLAIN ANALYZE (timed run)

## The story (map vs drive)

| Tool | What it does | Everyday picture |
|------|--------------|------------------|
| `EXPLAIN` | Shows the **planned** route | Ask GPS for directions (estimates) |
| `EXPLAIN ANALYZE` | **Runs** the query, then reports real cost | Drive the route, then see how long each turn took |

```
EXPLAIN            ->  estimated plan (does not time the full work the same way)
EXPLAIN ANALYZE    ->  execute + measure (actual timings / row counts in the tree)
```

Use **`EXPLAIN`** while learning the *shape* of a plan (`type`, `key`, `rows`).  
Use **`EXPLAIN ANALYZE`** when you want **evidence** on your laptop data: "Was the tight plan actually fast?"

> Needs MySQL **8.0.18+**. Most current installs support it.

## Important differences from plain EXPLAIN

| Topic | `EXPLAIN` | `EXPLAIN ANALYZE` |
|-------|-----------|-------------------|
| Runs the query? | Plan only (no full timed execution story) | **Yes** - it executes the statement to measure |
| Output style | Table of columns (`type`, `key`, `rows`, ...) | **Tree** of operators with timing text |
| `rows` meaning | Estimate | You also see **actual** rows examined / produced |
| Best for | "Which index might it use?" | "How expensive was it really?" |
| Caution | Cheap to run often | Can be slower on heavy queries - it does the work |

Class tip: start with `EXPLAIN`. When two plans look different, confirm with `EXPLAIN ANALYZE`.

## How to read the tree (what to hunt for)

MySQL prints a nested plan. You do not need every word. Scan for:

| Phrase / idea | Meaning |
|---------------|---------|
| `->` tree lines | Steps of the plan (children feed parents) |
| `cost=...` | Optimizer cost estimate for that step |
| `actual time=...` | Real time measured while running (often `first_row..last_row`) |
| `rows=...` (estimate) vs actual rows | Compare "thought" vs "did" |
| Index / table lookup names | Confirms whether an index path was used |
| Filter / Table scan wording | Full scan vs index access |

A tiny mental checklist:

1. Did it use an **index lookup** or a **table scan**?
2. Are **actual times** tiny (microseconds/ms) or large?
3. Do **actual rows** stay small for a selective filter?

## When to prefer which

| Situation | Prefer |
|-----------|--------|
| Learning `type` / `key` / `rows` | `EXPLAIN` |
| Comparing "indexed date" vs "broad freight" cost | both - shape then timing |
| Checking a rewrite (sargable range vs `EXTRACT(YEAR...)`) | `EXPLAIN ANALYZE` on both |
| Huge report query in production | be careful - ANALYZE executes it |

## What you will compare next

Run the same three questions as Chapter 3, but with timings:

1. Primary-key lookup (should be very cheap)
2. Indexed date filter (usually cheap)
3. Broad freight filter (often more work - scan many rows)

Keep the numbers in mind: **fast + few rows** vs **slower + many rows**.


### EXPLAIN ANALYZE for a primary-key lookup


In [4]:
EXPLAIN ANALYZE
SELECT *
FROM training.customers
WHERE customer_id = 1001;


-> Rows fetched before execution  (cost=0..0 rows=1) (actual time=0.00545..0.00549 rows=1 loops=1)


### EXPLAIN ANALYZE for an indexed date filter


In [5]:
EXPLAIN ANALYZE
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


-> Index lookup on orders using idx_orders_order_date (order_date=DATE'2023-06-15')  (cost=19.9 rows=57) (actual time=0.0707..0.116 rows=57 loops=1)


### EXPLAIN ANALYZE for a broad freight filter


**Story:** Same broad filter as before (`freight > 50`).

Watch for a **table scan** style step and a larger **actual time** / row count than the PK and date lookups.


In [6]:
EXPLAIN ANALYZE
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


-> Filter: (orders.freight > 50.00)  (cost=3940 rows=12997) (actual time=0.805..8.38 rows=40000 loops=1)
-> Table scan on orders  (cost=3940 rows=38996) (actual time=0.262..4.65 rows=40000 loops=1)


### How to compare the three ANALYZE results

After you run the cells above, ask:

1. **Primary key** - is actual time tiny and rows about 1?
2. **Indexed date** - still relatively small time / row count?
3. **Broad freight** - larger actual time and many more rows examined?

That is the whole point of `EXPLAIN ANALYZE`: the plan story from Chapter 3 becomes a **timed** story on your data.

```
Selective lookup     ->  small actual time, small actual rows
Broad scan           ->  larger actual time, many rows touched
```

If a selective filter still looks expensive, re-check `WHERE` shape (Chapter 5 - sargable filters).


---
# Chapter 5 - Sargable filters

**Sargable** means: write the `WHERE` so MySQL can use an index on that column.

Same question: *orders in 2023*

| Better (sargable) | Weaker |
|-------------------|--------|
| `order_date >= '2023-01-01' AND order_date < '2024-01-01'` | `EXTRACT(YEAR FROM order_date) = 2023` |

Rule: filter on the **column**, not on a **function of the column**.


### Better: date range


In [7]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2024-01-01';


+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys         | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | range | idx_orders_order_date | idx_orders_order_date | 3       | NULL | 19498 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+


### Weaker: `EXTRACT(YEAR ...)`


In [8]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE EXTRACT(YEAR FROM order_date) = 2023;


+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | index | NULL          | idx_orders_order_date | 3       | NULL | 38996 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+


Compare `type`, `key`, and `rows` above. Prefer the **range** form when an `order_date` index exists.


---
# Chapter 6 - Create, view, and drop an index

In class we mostly **read** plans with `EXPLAIN`. Here you also practice the three everyday index commands:

| Action | SQL idea |
|--------|----------|
| **View** | `SHOW INDEX FROM table_name;` |
| **Create** | `CREATE INDEX name ON table_name (column);` |
| **Drop** | `DROP INDEX name ON table_name;` |

## Safety rule for this course

Do **not** add or drop indexes on the shared sample tables `training.*` during class.

1. **View** indexes that already exist on `training.orders` (read-only).
2. **Create / view / drop** on a **temporary** copy - safe; it disappears when you disconnect.


### View indexes already on `training.orders`

`SHOW INDEX` lists each index, which columns it covers, and whether values must be unique.

Useful columns in the result:

| Column | Meaning |
|--------|---------|
| `Key_name` | Index name (`PRIMARY` is the primary key) |
| `Column_name` | Column inside that index |
| `Non_unique` | `0` = unique / primary; `1` = duplicates allowed |
| `Seq_in_index` | Column order inside a multi-column index |


In [11]:
SHOW INDEX FROM training.orders;


+--------+------------+-----------------------+--------------+-------------+-----------+-------------+----------+--------+------+------------+---------+---------------+---------+------------+
| Table  | Non_unique | Key_name              | Seq_in_index | Column_name | Collation | Cardinality | Sub_part | Packed | Null | Index_type | Comment | Index_comment | Visible | Expression |
+--------+------------+-----------------------+--------------+-------------+-----------+-------------+----------+--------+------+------------+---------+---------------+---------+------------+
| orders |          0 | PRIMARY               |            1 | order_id    | A         |       38996 |     NULL |   NULL |      | BTREE      |         |               | YES     | NULL       |
| orders |          1 | fk_ord_cust           |            1 | customer_id | A         |        4836 |     NULL |   NULL | YES  | BTREE      |         |               | YES     | NULL       |
| orders |          1 | fk_ord_emp      

### Create, view, then drop an index (temporary table)

Same idea as a scratch notebook page:

1. Copy a small slice of orders into `temp_idx_lab`
2. `CREATE INDEX` on `customer_id`
3. `SHOW INDEX` to confirm it is there
4. `DROP INDEX` to remove it
5. `SHOW INDEX` again - the extra index is gone

All of this stays in **one session** so the temporary table still exists.


In [12]:
DROP TEMPORARY TABLE IF EXISTS temp_idx_lab;

CREATE TEMPORARY TABLE temp_idx_lab AS
SELECT order_id, customer_id, order_date, status, freight
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2023-02-01';

-- Create
CREATE INDEX idx_temp_customer_id
  ON temp_idx_lab (customer_id);

-- View (look for Key_name = idx_temp_customer_id)
SHOW INDEX FROM temp_idx_lab;

-- Drop
DROP INDEX idx_temp_customer_id ON temp_idx_lab;

-- View again (that index name should be gone)
SHOW INDEX FROM temp_idx_lab;


+--------------+------------+----------------------+--------------+-------------+-----------+-------------+----------+--------+------+------------+---------+---------------+---------+------------+
| Table        | Non_unique | Key_name             | Seq_in_index | Column_name | Collation | Cardinality | Sub_part | Packed | Null | Index_type | Comment | Index_comment | Visible | Expression |
+--------------+------------+----------------------+--------------+-------------+-----------+-------------+----------+--------+------+------------+---------+---------------+---------+------------+
| temp_idx_lab |          1 | idx_temp_customer_id |            1 | customer_id | A         |        1844 |     NULL | NULL   | YES  | BTREE      |         |               | YES     | NULL       |
+--------------+------------+----------------------+--------------+-------------+-----------+-------------+----------+--------+------+------------+---------+---------------+---------+------------+


### Use the index after you create it

Creating an index is only half the story. Next we create a temp table again, add an index, and prove it with `EXPLAIN` / `EXPLAIN ANALYZE`.


### Build temp table, add index, then explain the lookup

This one cell does four things **in the same session** (needed so the temp table still exists):

1. `CREATE TEMPORARY TABLE temp_orders AS ...` - small copy (Q1 2023)
2. `CREATE INDEX ... ON temp_orders (customer_id)` - add the shortcut
3. `EXPLAIN ... WHERE customer_id = 1001` - plan table (`type` / `key` / `rows`)
4. `EXPLAIN ANALYZE ...` - timed tree for the same lookup

What you hope to see on `EXPLAIN`:

| Column | Likely healthy result |
|--------|------------------------|
| `table` | `temp_orders` |
| `type` | `ref` (index lookup) |
| `key` | `temp_orders_customer_id_idx` |
| `rows` | small (near 1 for this customer in the slice) |

Then confirm with `EXPLAIN ANALYZE` that the lookup is cheap.


In [13]:
CREATE TEMPORARY TABLE temp_orders AS
SELECT order_id, customer_id, order_date, status, freight
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2023-04-01';

CREATE INDEX temp_orders_customer_id_idx
  ON temp_orders (customer_id);

EXPLAIN
SELECT *
FROM temp_orders
WHERE customer_id = 1001;

EXPLAIN ANALYZE
SELECT *
FROM temp_orders
WHERE customer_id = 1001;


1	SIMPLE	temp_orders	NULL	ref	temp_orders_customer_id_idx	temp_orders_customer_id_idx	5	const	1	100.00	NULL
-> Index lookup on temp_orders using temp_orders_customer_id_idx (customer_id=1001)  (cost=0.35 rows=1) (actual time=0.00732..0.00792 rows=1 loops=1)


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Sargable filters</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
